# "THE PRICE IS RIGHT" Capstone Project:

In this module, we'll build a model that predicts how much something costs from a description, based on a scrape of Amazon data

## Order of play:

DAY 1: Data Curation

DAY 2: Data Pre-processing

DAY 3: Evaluation, Baselines, Traditional ML

DAY 4: Deep Learning and LLMs

DAY 5: Fine-tuning a Frontier Model

### DAY 1: Data Curation

Today we'll scrub our dataset and curate our data.

The dataset is here:
https://huggingface.co/datasets/McAuley-Lab/Amazon-Reviews-2023

And the folder with all the product datasets is here:
https://huggingface.co/datasets/McAuley-Lab/Amazon-Reviews-2023/tree/main/raw/meta_categories

In [5]:
# Imports:
import os
from dotenv import load_dotenv
from huggingface_hub import login
from datasets import load_dataset
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm
import numpy as np
import random
from pricer.items import Item
from pricer.parser import parse
load_dotenv(override= True)

True

In [6]:
# Logging in to Hugging Face:
hf_token = os.getenv('HF_TOKEN')
login(token = hf_token, add_to_git_credential= True)

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


### Loading in Dataset:

In the next cell, we load in the dataset from huggingface.

If this gives you an error like "trust_remote_code is no longer supported", then please run this command in a new cell: `!pip install --upgrade datasets==3.6.0` and then restart the Kernel, and try again.

In [7]:
dataset= load_dataset('McAuley-Lab/Amazon-Reviews-2023',
                      'raw_meta_Appliances',
                      split= 'full',
                      trust_remote_code= True)

README.md: 0.00B [00:00, ?B/s]

C:\Users\shail\anaconda3\envs\applied_llm_engineering\lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\shail\.cache\huggingface\hub\datasets--McAuley-Lab--Amazon-Reviews-2023. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


Amazon-Reviews-2023.py: 0.00B [00:00, ?B/s]

raw/meta_categories/meta_Appliances.json(…):   0%|          | 0.00/285M [00:00<?, ?B/s]

Generating full split: 0 examples [00:00, ? examples/s]

In [8]:
print(f'Number of Appliances: {len(dataset)}')

Number of Appliances: 94327


In [11]:
# Investigating a Particular Datapoint:
dataset[10]

{'main_category': 'Tools & Home Improvement',
 'title': 'WP67003405 67003405 Door Pivot Block - Compatible Kenmore KitchenAid Maytag Whirlpool Refrigerator - Replaces AP6010352 8208254 PS11743531 - Quick DIY Repair Solution',
 'average_rating': 4.1,
 'rating_number': 4,
 'features': ['WP67003405 Pivot Block For Vernicle Mullion Strip On Door - A high-quality exact equivalent for part numbers AP6010352, 67003405, 1025322, 12698403, 67003194, 8208254, and PS11743531.',
  'Compatibility with major brands - WP67003405 Door Guide is compatible with Whirlpool, Amana, Dacor, Gaggenau, Hardwick, Jenn-Air, Kenmore, KitchenAid, and Maytag.',
  "Quick DIY repair - WP67003405 Refrigerator Door Guide Pivot Block Replacement will help if your appliance door doesn't open or close. Wear work gloves to protect your hands during the repair process.",
  'Attentive support - If you are uncertain about whether the block fits your refrigerator, we will help. We generally put forth a valiant effort to guaran